# Recommend runs from redundancy scores

Turns the `layer_priority.py` score files into candidate rows for an
`experiments_*.csv`, in the exact 9-column schema the pipeline expects
(`dataset,encoder,skip,mlp_skip,attn_skip,head_dict,skip_translator,mlp_mode,attn_mode`).

**`skip` convention (SkipModel):** a pair `(skip_from, skip_to)` keeps block
`skip_from`, DROPS blocks `skip_from+1 .. skip_to`, and bridges `out(skip_from) ->
out(skip_to)`. Both indices must be in `0 .. num_layers-1`, so `skip_to <=
num_layers-1` — a pair like `(8, 12)` on a 12-block model is illegal. To drop a
single block `j`, write `(j-1, j)`.

**Nothing here is automatic** — every knob is exposed so you can eyeball the scores
(use `analysis.ipynb`) and adjust before writing the CSV. Always keep the baseline
row: the pipeline needs it to compute `delta_acc`.


In [21]:
import ast, csv
from pathlib import Path

import numpy as np
import pandas as pd

# ---- what to read and emit ----
OUTPUT_DIR = Path("../outputs")
STEM = "raddino_chestmnist"
DATASET = "chestmnist"
ENCODER = "microsoft/rad-dino"
OUT_CSV = Path("../../configs/experiments.csv")

# ---- knobs ----
N_SINGLE_BLOCKS = 3     # lowest-BI single blocks to try, dropped as skip=[(j-1, j)]
MAX_SPAN        = 4     # try best placement for dropping 2..MAX_SPAN contiguous blocks
N_MLP           = 3     # lowest-BI MLP layers -> one combined mlp_skip candidate (+ individually)
N_ATTN          = 3     # lowest-BI attention layers -> one combined attn_skip candidate (+ individually)
CKA_IDENTITY    = 0.90  # bridged-endpoint CKA above this -> skip_translator=identity, else linear
HEAD_THRESHOLD  = 0.90  # head-JSD similarity above this -> prune the higher-index head
# --------------------------------

def _p(s): return OUTPUT_DIR / f"{STEM}_{s}"

def load_block_scores(path):
    # Two sections in one file; switch on the header token (robust to missing blank line).
    block, span, mode = {}, {}, None
    for r in csv.reader(open(path)):
        if not r or not r[0]:
            continue
        if r[0] == "block":
            mode = "block"; continue
        if r[0] == "blocks_dropped":
            mode = "span"; continue
        if mode == "block":
            block[int(r[0])] = float(r[1])
        elif mode == "span":
            span[int(r[0])] = (int(r[1]), int(r[2]), float(r[3]))
    return block, span

block_bi, span = load_block_scores(_p("block_scores.csv"))
cka = np.load(_p("cka.npy"))
sub_path = _p("sublayer_scores.csv")
sub = {int(r[0]): tuple(map(float, r[1:])) for r in list(csv.reader(open(sub_path)))[1:]} if sub_path.exists() else {}
head_path = _p("head_similarity.npy")
head_sim = np.load(head_path) if head_path.exists() else None
L = len(block_bi)
print(f"Loaded {STEM}: {L} blocks, sublayer={bool(sub)}, heads={head_sim is not None}")

Loaded raddino_chestmnist: 12 blocks, sublayer=True, heads=True


### Row builder
`skip` = list of `(skip_from, skip_to)` tuples; `mlp_skip`/`attn_skip` = int lists; `head_dict` = `{layer: [heads_to_keep]}`. `translator_for_span` picks identity/linear from the CKA between the two hidden states the skip bridges (`skip_from+1`, `skip_to+1`).

In [22]:
def row(skip=None, mlp_skip=None, attn_skip=None, head_dict=None,
        skip_translator="identity", mlp_mode="identity", attn_mode="identity", note=""):
    return {
        "dataset": DATASET, "encoder": ENCODER,
        "skip": repr(skip or []),
        "mlp_skip": repr(mlp_skip or []),
        "attn_skip": repr(attn_skip or []),
        "head_dict": repr(head_dict or {}),
        "skip_translator": skip_translator,
        "mlp_mode": mlp_mode, "attn_mode": attn_mode,
        "_note": note,
    }

def translator_for_span(skip_from, skip_to):
    # skip bridges hidden state (skip_from+1) -> (skip_to+1)
    return "identity" if cka[skip_from + 1, skip_to + 1] >= CKA_IDENTITY else "linear"

def head_keep_dict(threshold):
    # greedy within-layer prune (same rule as head_priority.py)
    if head_sim is None: return {}
    Lh, H, _ = head_sim.shape
    out = {}
    for l in range(Lh):
        remove = set()
        for i in range(H):
            if i in remove: continue
            for j in range(i + 1, H):
                if head_sim[l, i, j] > threshold: remove.add(j)
        out[l] = sorted(set(range(H)) - remove)
    return out

### Assemble candidates
Baseline first, then single blocks (`(j-1, j)`), best spans, MLP, attention, heads, and a combined 'safest of each'. Block 0 is skipped for single-block drops (no `skip_from = -1`). Tune the knobs above and re-run.

In [23]:
cands = [row(note="baseline")]

# --- single whole blocks (lowest BI); drop block j via skip=(j-1, j), so skip block 0 ---
single = [j for j in sorted(block_bi, key=block_bi.get) if j >= 1][:N_SINGLE_BLOCKS]
for j in single:
    cands.append(row(skip=[(j - 1, j)], skip_translator=translator_for_span(j - 1, j),
                     note=f"drop block {j}  BI={block_bi[j]:.3f}  cka={cka[j, j + 1]:.2f}"))

# --- best contiguous multi-block spans ---
for k in range(2, MAX_SPAN + 1):
    if k in span:
        s, e, d = span[k]
        cands.append(row(skip=[(s, e)], skip_translator=translator_for_span(s, e),
                         note=f"drop {k} blocks {s + 1}..{e}  angdist={d:.3f}  cka={cka[s + 1, e + 1]:.2f}"))

# --- MLP sub-blocks ---
if sub:
    mlp_rank = sorted(sub, key=lambda i: sub[i][2])
    top_mlp = mlp_rank[:N_MLP]
    for i in top_mlp:
        cands.append(row(mlp_skip=[i], mlp_mode="identity", note=f"mlp {i}  BI={sub[i][2]:.3f}"))
    cands.append(row(mlp_skip=sorted(top_mlp), mlp_mode="identity", note=f"mlp combined {sorted(top_mlp)}"))

    # --- attention sub-blocks ---
    attn_rank = sorted(sub, key=lambda i: sub[i][0])
    top_attn = attn_rank[:N_ATTN]
    for i in top_attn:
        cands.append(row(attn_skip=[i], attn_mode="identity", note=f"attn {i}  BI={sub[i][0]:.3f}"))
    cands.append(row(attn_skip=sorted(top_attn), attn_mode="identity", note=f"attn combined {sorted(top_attn)}"))

# --- head pruning ---
hd = head_keep_dict(HEAD_THRESHOLD)
if hd:
    pruned = sum(head_sim.shape[1] - len(v) for v in hd.values())
    cands.append(row(head_dict=hd, note=f"heads @thr={HEAD_THRESHOLD}  ({pruned} pruned)"))

# --- combined: safest single block + safest MLP + safest attn + heads ---
if sub and single:
    b0 = single[0]
    cands.append(row(skip=[(b0 - 1, b0)], mlp_skip=[mlp_rank[0]], attn_skip=[attn_rank[0]],
                     head_dict=hd, skip_translator=translator_for_span(b0 - 1, b0),
                     mlp_mode="identity", attn_mode="identity", note="combined safest-of-each"))

df = pd.DataFrame(cands)
df[["_note"] + [c for c in df.columns if c != "_note"]]

,_note,dataset,encoder,skip,mlp_skip,attn_skip,head_dict,skip_translator,mlp_mode,attn_mode
0,baseline,chestmnist,microsoft/rad-dino,[],[],[],{},identity,identity,identity
1,drop block 10 BI=0.026 cka=0.96,chestmnist,microsoft/rad-dino,"[(9, 10)]",[],[],{},identity,identity,identity
2,drop block 9 BI=0.034 cka=0.96,chestmnist,microsoft/rad-dino,"[(8, 9)]",[],[],{},identity,identity,identity
3,drop block 8 BI=0.041 cka=0.97,chestmnist,microsoft/rad-dino,"[(7, 8)]",[],[],{},identity,identity,identity
4,drop 2 blocks 9..10 angdist=0.118 cka=0.90,chestmnist,microsoft/rad-dino,"[(8, 10)]",[],[],{},linear,identity,identity
5,drop 3 blocks 9..11 angdist=0.154 cka=0.78,chestmnist,microsoft/rad-dino,"[(8, 11)]",[],[],{},linear,identity,identity
6,drop 4 blocks 8..11 angdist=0.178 cka=0.77,chestmnist,microsoft/rad-dino,"[(7, 11)]",[],[],{},linear,identity,identity
7,mlp 11 BI=0.017,chestmnist,microsoft/rad-dino,[],[11],[],{},identity,identity,identity
8,mlp 10 BI=0.018,chestmnist,microsoft/rad-dino,[],[10],[],{},identity,identity,identity
9,mlp 9 BI=0.026,chestmnist,microsoft/rad-dino,[],[9],[],{},identity,identity,identity


### Write the config CSV
Drops the `_note` helper column and writes the 9-column file, ready for `run_pipeline_row_by_row.sh` via `CONFIG_CSV=`. Review the table above first. The round-trip check also asserts every skip pair is in range `0..L-1`.

In [24]:
schema = ["dataset", "encoder", "skip", "mlp_skip", "attn_skip",
          "head_dict", "skip_translator", "mlp_mode", "attn_mode"]
out = df[schema].copy()
out.to_csv(OUT_CSV, index=False, quoting=csv.QUOTE_MINIMAL)
print(f"Wrote {len(out)} rows -> {OUT_CSV.resolve()}")

# round-trip check: every literal parses back AND every skip index is a valid layer-output key
for col in ["skip", "mlp_skip", "attn_skip", "head_dict"]:
    for v in out[col]:
        ast.literal_eval(v)
for v in out["skip"]:
    for a, b in ast.literal_eval(v):
        assert 0 <= a < b <= L - 1, f"illegal skip ({a}, {b}) for {L}-block model"
print(f"All literals parse; all skip pairs within 0..{L - 1}.")
print("\nNext: CONFIG_CSV=%s bash src/run_scripts/run_pipeline_row_by_row.sh" % OUT_CSV)

Wrote 17 rows -> /Users/mabelwylie/Documents/toast-extensions/src/configs/experiments.csv
All literals parse; all skip pairs within 0..11.

Next: CONFIG_CSV=../../configs/experiments.csv bash src/run_scripts/run_pipeline_row_by_row.sh
